In [1]:
import pickle

# Проверяем оригинальную ALS модель
with open('artifacts/als_model.pkl', 'rb') as f:
    orig_als = pickle.load(f)
print("Type:", type(orig_als))
print("Dir:", [x for x in dir(orig_als) if not x.startswith('_')])

if hasattr(orig_als, 'user_factors'):
    print(f"User factors: {orig_als.user_factors.shape}")
    print(f"Item factors: {orig_als.item_factors.shape}")

with open('artifacts/als_model_data.pkl', 'rb') as f:
    orig_data = pickle.load(f)
print(f"\nals_model_data type: {type(orig_data)}")
if isinstance(orig_data, dict):
    print(f"Keys: {orig_data.keys()}")
    for k, v in orig_data.items():
        if hasattr(v, 'shape'):
            print(f"  {k}: {v.shape}")
        elif isinstance(v, dict):
            print(f"  {k}: dict with {len(v)} items")
        else:
            print(f"  {k}: {type(v)}")


Type: <class 'implicit.cpu.als.AlternatingLeastSquares'>
Dir: ['XtX', 'YtY', 'alpha', 'calculate_training_loss', 'cg_steps', 'dtype', 'explain', 'factors', 'fit', 'fit_callback', 'item_factors', 'item_norms', 'iterations', 'load', 'num_threads', 'partial_fit_items', 'partial_fit_users', 'random_state', 'rank_items', 'recalculate_item', 'recalculate_user', 'recommend', 'recommend_all', 'regularization', 'save', 'similar_items', 'similar_users', 'solver', 'to_gpu', 'use_cg', 'use_native', 'user_factors', 'user_norms']
User factors: (5000, 33)
Item factors: (24700, 33)

als_model_data type: <class 'dict'>
Keys: dict_keys(['model', 'item2idx', 'idx2item'])
  model: <class 'implicit.cpu.als.AlternatingLeastSquares'>
  item2idx: dict with 18953 items
  idx2item: dict with 18953 items


In [2]:
import pandas as pd
import numpy as np
import pickle
import ast
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# LOAD
# ============================================================
history = pd.read_parquet('artifacts/history_interactions.parquet')
games = pd.read_csv('data/game_details.csv')
users = pd.read_csv('data/unique_users.csv')
item_feat_mh = pd.read_parquet('artifacts/item_features_multihot.parquet')

# v2
item_pop = pd.read_parquet('artifacts/v2_item_pop.parquet')
item_dev_agg = pd.read_parquet('artifacts/v2_item_dev.parquet')
item_pub_agg = pd.read_parquet('artifacts/v2_item_pub.parquet')
user_stats = pd.read_parquet('artifacts/v2_user_stats.parquet')
user_plat = pd.read_parquet('artifacts/v2_user_plat.parquet')
user_free = pd.read_parquet('artifacts/v2_user_free.parquet')
user_dev = pd.read_parquet('artifacts/v2_user_dev.parquet')
user_pub = pd.read_parquet('artifacts/v2_user_pub.parquet')
dev_per_appid = pd.read_parquet('artifacts/v2_dev_per_appid.parquet')
pub_per_appid = pd.read_parquet('artifacts/v2_pub_per_appid.parquet')

# v3
item_dev_te = pd.read_parquet('artifacts/v3_item_dev_te.parquet')
item_pub_te = pd.read_parquet('artifacts/v3_item_pub_te.parquet')
item_genre_te = pd.read_parquet('artifacts/v3_item_genre_te.parquet')
user_dev_te = pd.read_parquet('artifacts/v3_user_dev_te.parquet')
dev_exploded = pd.read_parquet('artifacts/v3_dev_exploded.parquet')

# v4
uf_df = pd.read_parquet('artifacts/v4_user_factors.parquet')
if_df = pd.read_parquet('artifacts/v4_item_factors.parquet')
item_factors_normed = np.load('artifacts/v4_item_factors_normed.npy')
user_centroids = np.load('artifacts/v4_user_centroids.npy')
user_centroid_meta = pd.read_parquet('artifacts/v4_user_centroid_meta.parquet')

with open('artifacts/v4_user_top_conf.pkl', 'rb') as f:
    user_top_conf = pickle.load(f)
with open('artifacts/v4_user2idx.pkl', 'rb') as f:
    user2idx = pickle.load(f)
with open('artifacts/v4_item2idx.pkl', 'rb') as f:
    item2idx = pickle.load(f)

N_FACTORS = 128
uf_cols = [f'uf_{i}' for i in range(N_FACTORS)]
if_cols = [f'if_{i}' for i in range(N_FACTORS)]

mh_genre_cols = [c for c in item_feat_mh.columns if c.startswith('genres_')]
mh_cat_cols = [c for c in item_feat_mh.columns if c.startswith('categories_')]
mh_cols = mh_genre_cols + mh_cat_cols

# Precompute
max_time = 1776010854
user_profile = users[['steamid', 'loccountrycode', 'timecreated']].copy()
user_profile['loccountrycode'] = user_profile['loccountrycode'].fillna('UNKNOWN')
user_profile['account_age_days'] = ((max_time - user_profile['timecreated'].fillna(max_time)) / 86400).clip(lower=0)
country_counts = user_profile['loccountrycode'].value_counts()
user_profile['country_freq'] = user_profile['loccountrycode'].map(country_counts).fillna(0).astype(int)
user_profile = user_profile.drop(columns=['timecreated'])

game_type = games[['appid', 'type']].copy()
game_type['type'] = game_type['type'].fillna('unknown')

print("All loaded!")

# ============================================================
# 1. NEGATIVE SUBSAMPLING: top-K by ALS + all positives
# ============================================================
print("\n" + "="*60)
print("[1] Negative subsampling: top-100 by ALS rank + all positives")
print("="*60)

TOP_K_CANDIDATES = 100

def subsample_candidates(base_path, target_path):
    df = pd.read_parquet(base_path)
    df = df.rename(columns={'score': 'als_score', 'rank': 'als_rank'})
    
    # Load targets
    targets = pd.read_parquet(target_path) if target_path else None
    
    # Mark positives
    if targets is not None:
        future_targets = pd.read_parquet('artifacts/history_interactions.parquet')  # not needed, targets in base
    
    # targets are already merged in train/test
    before = len(df)
    
    # Keep: top-K by als_rank OR positive
    top_k_mask = df['als_rank'] <= TOP_K_CANDIDATES
    
    # We need target info. Let's load from the already-assembled files
    return df[top_k_mask]

# For train and test: filter to top-100 als_rank
# But we need target info. Let's use the existing train/test with target
train_base = pd.read_parquet('artifacts/train_reranker_base.parquet')
test_base = pd.read_parquet('artifacts/test_reranker_base.parquet')

# Rename
train_base = train_base.rename(columns={'score': 'als_score', 'rank': 'als_rank'})
test_base = test_base.rename(columns={'score': 'als_score', 'rank': 'als_rank'})

def subsample(df, top_k=TOP_K_CANDIDATES):
    """Keep top-K by als_rank + ALL positives per user"""
    is_positive = df['target'] > 0
    is_top_k = df['als_rank'] <= top_k
    keep = is_positive | is_top_k
    result = df[keep].copy()
    return result

train_sub = subsample(train_base, TOP_K_CANDIDATES)
test_sub = subsample(test_base, TOP_K_CANDIDATES)

print(f"Train: {len(train_base)} → {len(train_sub)} ({len(train_sub)/len(train_base)*100:.1f}%)")
print(f"Test:  {len(test_base)} → {len(test_sub)} ({len(test_base)/len(test_base)*100:.1f}%)")
print(f"Train positive rate: {(train_sub['target']>0).mean():.4f} (was {(train_base['target']>0).mean():.4f})")
print(f"Test positive rate:  {(test_sub['target']>0).mean():.4f}")

# ============================================================
# 2. PCA ON FACTOR PRODUCTS
# ============================================================
print("\n" + "="*60)
print("[2] PCA on ALS factor products")
print("="*60)

N_PCA = 24

# Compute all factor products on FULL data (train+test) for consistent PCA
all_steamids = np.union1d(train_sub['steamid'].unique(), test_sub['steamid'].unique())
all_appids = np.union1d(train_sub['appid'].unique(), test_sub['appid'].unique())

# Build factor product matrix for fitting PCA
# Use train only to fit PCA
train_with_factors = train_sub[['steamid', 'appid']].merge(uf_df, on='steamid', how='left')
train_with_factors = train_with_factors.merge(if_df, on='appid', how='left')
for c in uf_cols + if_cols:
    train_with_factors[c] = train_with_factors[c].fillna(0)

uf_train = train_with_factors[uf_cols].values.astype(np.float32)
if_train = train_with_factors[if_cols].values.astype(np.float32)
products_train = uf_train * if_train

pca = PCA(n_components=N_PCA, random_state=42)
pca.fit(products_train)
print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.4f} ({N_PCA} components)")
print(f"Per component: {pca.explained_variance_ratio_[:5].round(4)}")

# Save PCA model
with open('artifacts/v5_pca.pkl', 'wb') as f:
    pickle.dump(pca, f)

del train_with_factors, uf_train, if_train, products_train

# ============================================================
# 3. CONDITIONAL USER FEATURES (КЛЮЧЕВОЕ!)
# ============================================================
print("\n" + "="*60)
print("[3] Conditional user features")
print("="*60)

# User's avg target and playtime BY GENRE
# For each genre, what's the user's average engagement?
hist_with_genres = history[['steamid', 'appid', 'playtime_forever', 'target']].merge(
    item_feat_mh[['appid'] + mh_genre_cols], on='appid', how='inner'
)

# Melt genres
genre_records = []
for col in mh_genre_cols:
    genre_name = col  # e.g., 'genres_Action'
    subset = hist_with_genres[hist_with_genres[col] == 1][['steamid', 'playtime_forever', 'target']].copy()
    subset['genre'] = genre_name
    genre_records.append(subset)

user_genre_stats = pd.concat(genre_records, ignore_index=True)
user_genre_agg = user_genre_stats.groupby(['steamid', 'genre']).agg(
    cond_genre_avg_target=('target', 'mean'),
    cond_genre_avg_pt=('playtime_forever', 'mean'),
    cond_genre_n_games=('target', 'count'),
).reset_index()

print(f"User-Genre conditional stats: {len(user_genre_agg)} pairs")

# Similarly for categories (top-5 most important)
top_cat_cols = ['categories_Single-player', 'categories_Multi-player', 
                'categories_Co-op', 'categories_PvP', 'categories_Steam_Achievements']
top_cat_cols = [c for c in top_cat_cols if c in mh_cat_cols]

hist_with_cats = history[['steamid', 'appid', 'playtime_forever', 'target']].merge(
    item_feat_mh[['appid'] + top_cat_cols], on='appid', how='inner'
)

cat_records = []
for col in top_cat_cols:
    subset = hist_with_cats[hist_with_cats[col] == 1][['steamid', 'playtime_forever', 'target']].copy()
    subset['category'] = col
    cat_records.append(subset)

user_cat_stats = pd.concat(cat_records, ignore_index=True)
user_cat_agg = user_cat_stats.groupby(['steamid', 'category']).agg(
    cond_cat_avg_target=('target', 'mean'),
    cond_cat_n_games=('target', 'count'),
).reset_index()

print(f"User-Category conditional stats: {len(user_cat_agg)} pairs")

# ============================================================
# 4. PLAYTIME-WEIGHTED AFFINITY
# ============================================================
print("\n[4] Playtime-weighted affinity...")

hist_mh = history[['steamid', 'appid', 'playtime_forever']].merge(
    item_feat_mh[['appid'] + mh_cols], on='appid', how='inner'
)
hist_mh['log_pt'] = np.log1p(hist_mh['playtime_forever']).astype('float32')

user_game_counts = hist_mh.groupby('steamid').size().reset_index(name='_cnt')
user_aff_count = hist_mh.groupby('steamid')[mh_cols].sum().reset_index()
user_aff_count = user_aff_count.merge(user_game_counts, on='steamid')
for col in mh_cols:
    user_aff_count[f'uaff_{col}'] = (user_aff_count[col] / user_aff_count['_cnt']).astype('float32')
uaff_count_cols = [f'uaff_{c}' for c in mh_cols]
user_aff_count = user_aff_count[['steamid'] + uaff_count_cols]

# Weighted
for col in mh_cols:
    hist_mh[f'w_{col}'] = hist_mh[col].values * hist_mh['log_pt'].values
w_cols = [f'w_{c}' for c in mh_cols]
user_aff_wt = hist_mh.groupby('steamid')[w_cols].sum().reset_index()
user_total_wt = hist_mh.groupby('steamid')['log_pt'].sum().reset_index().rename(columns={'log_pt': '_tw'})
user_aff_wt = user_aff_wt.merge(user_total_wt, on='steamid')
for col in mh_cols:
    user_aff_wt[f'uwaff_{col}'] = (user_aff_wt[f'w_{col}'] / user_aff_wt['_tw'].clip(lower=0.001)).astype('float32')
uwaff_cols = [f'uwaff_{c}' for c in mh_cols]
user_aff_wt = user_aff_wt[['steamid'] + uwaff_cols]

print("All precomputed!")

# ============================================================
# ASSEMBLY FUNCTION v5
# ============================================================
def assemble_v5(df):
    N = len(df)
    print(f"\n  Assembling v5, rows={N}")
    
    # ---- ALS SCORE FEATURES ----
    print("    [1] ALS score features...")
    df['als_score_log'] = np.log1p(df['als_score'])
    df['als_rank_inv'] = 1.0 / df['als_rank']
    df['als_rank_inv_sqrt'] = 1.0 / np.sqrt(df['als_rank'])
    df['als_rank_norm'] = df['als_rank'] / 300.0
    
    als_g = df.groupby('steamid')['als_score'].agg(['mean', 'std', 'max', 'min']).reset_index()
    als_g.columns = ['steamid', '_am', '_as', '_ax', '_an']
    df = df.merge(als_g, on='steamid')
    df['als_score_zscore'] = (df['als_score'] - df['_am']) / df['_as'].clip(lower=0.001)
    df['als_score_minmax'] = (df['als_score'] - df['_an']) / (df['_ax'] - df['_an']).clip(lower=0.001)
    df.drop(columns=['_am', '_as', '_ax', '_an'], inplace=True)
    
    # ---- ALS PCA FACTOR PRODUCTS (24 components) ----
    print("    [2] ALS PCA factor products...")
    df = df.merge(uf_df, on='steamid', how='left')
    df = df.merge(if_df, on='appid', how='left')
    for c in uf_cols + if_cols:
        df[c] = df[c].fillna(0)
    
    uf_vals = df[uf_cols].values.astype(np.float32)
    if_vals = df[if_cols].values.astype(np.float32)
    
    # Dot, cosine
    df['als_dot'] = np.sum(uf_vals * if_vals, axis=1)
    uf_n = np.linalg.norm(uf_vals, axis=1)
    if_n = np.linalg.norm(if_vals, axis=1)
    df['uf_norm'] = uf_n
    df['if_norm'] = if_n
    df['als_cosine'] = df['als_dot'] / (uf_n * if_n + 1e-8)
    
    # PCA on element-wise products
    products = uf_vals * if_vals
    pca_features = pca.transform(products)
    for i in range(N_PCA):
        df[f'pca_{i}'] = pca_features[:, i].astype(np.float32)
    
    # Also keep top-8 raw products by variance (for the model to learn non-linear patterns)
    product_var = np.var(products, axis=0)
    top8_dims = np.argsort(-product_var)[:8]
    for rank, dim in enumerate(top8_dims):
        df[f'uf_x_if_top{rank}'] = products[:, dim]
    
    df.drop(columns=uf_cols + if_cols, inplace=True)
    
    # ---- ITEM-ITEM SIMILARITY ----
    print("    [3] Item-item similarity...")
    steamids = df['steamid'].values
    appids = df['appid'].values
    
    user_idx_arr = np.array([user2idx.get(s, -1) for s in steamids], dtype=np.int32)
    item_idx_arr = np.array([item2idx.get(a, -1) for a in appids], dtype=np.int32)
    
    # Centroid sim (vectorized)
    valid_mask = (user_idx_arr >= 0) & (item_idx_arr >= 0)
    centroid_sim = np.zeros(N, dtype=np.float32)
    if valid_mask.any():
        centroid_sim[valid_mask] = np.sum(
            user_centroids[user_idx_arr[valid_mask]] * item_factors_normed[item_idx_arr[valid_mask]], axis=1
        )
    df['centroid_sim'] = centroid_sim
    
    # Top-K sim (batched)
    print("      Top-K similarity...")
    max_sim_arr = np.zeros(N, dtype=np.float32)
    avg_sim_arr = np.zeros(N, dtype=np.float32)
    
    df['_row_idx'] = np.arange(N)
    for sid, indices in df.groupby('steamid')['_row_idx'].apply(list).items():
        if sid not in user_top_conf:
            continue
        top_item_idx = user_top_conf[sid]
        if len(top_item_idx) == 0:
            continue
        cand_appids = appids[indices]
        cand_item_idx = np.array([item2idx.get(a, -1) for a in cand_appids])
        valid = cand_item_idx >= 0
        if not valid.any():
            continue
        sim_matrix = item_factors_normed[top_item_idx] @ item_factors_normed[cand_item_idx[valid]].T
        valid_indices = np.array(indices)[valid]
        max_sim_arr[valid_indices] = sim_matrix.max(axis=0)
        avg_sim_arr[valid_indices] = sim_matrix.mean(axis=0)
    
    df['max_sim_top10'] = max_sim_arr
    df['avg_sim_top10'] = avg_sim_arr
    df['sim_spread'] = max_sim_arr - avg_sim_arr
    df.drop(columns=['_row_idx'], inplace=True)
    
    # ---- USER FEATURES ----
    print("    [4] User features...")
    df = df.merge(user_profile, on='steamid', how='left')
    df = df.merge(user_stats, on='steamid', how='left')
    df = df.merge(user_plat, on='steamid', how='left')
    df = df.merge(user_free, on='steamid', how='left')
    df = df.merge(user_centroid_meta, on='steamid', how='left')
    
    # ---- ITEM FEATURES ----
    print("    [5] Item features...")
    item_compact = item_feat_mh[['appid', 'is_free', 'recommendations_log', 'age_years', 'platforms_count']].copy()
    df = df.merge(item_compact, on='appid', how='left')
    df = df.merge(item_pop, on='appid', how='left')
    df = df.merge(item_dev_agg, on='appid', how='left')
    df = df.merge(item_pub_agg, on='appid', how='left')
    df = df.merge(item_dev_te, on='appid', how='left')
    df = df.merge(item_pub_te, on='appid', how='left')
    df = df.merge(item_genre_te, on='appid', how='left')
    df = df.merge(game_type, on='appid', how='left')
    df['type'] = df['type'].fillna('unknown')
    
    # ---- GENRE/CATEGORY MATCH ----
    print("    [6] Genre/Category match...")
    df = df.merge(user_aff_count, on='steamid', how='left')
    df = df.merge(user_aff_wt, on='steamid', how='left')
    item_mh = item_feat_mh[['appid'] + mh_cols]
    df = df.merge(item_mh, on='appid', how='left')
    
    genre_dot = np.zeros(N, dtype='float32')
    genre_wdot = np.zeros(N, dtype='float32')
    u_gn = np.zeros(N, dtype='float32')
    i_gn = np.zeros(N, dtype='float32')
    uw_gn = np.zeros(N, dtype='float32')
    cat_dot = np.zeros(N, dtype='float32')
    cat_wdot = np.zeros(N, dtype='float32')
    u_cn = np.zeros(N, dtype='float32')
    i_cn = np.zeros(N, dtype='float32')
    
    for col in mh_genre_cols:
        iv = df[col].fillna(0).values.astype('float32')
        uv = df[f'uaff_{col}'].fillna(0).values.astype('float32')
        uwv = df[f'uwaff_{col}'].fillna(0).values.astype('float32')
        genre_dot += uv * iv; genre_wdot += uwv * iv
        u_gn += uv**2; i_gn += iv**2; uw_gn += uwv**2
    
    for col in mh_cat_cols:
        iv = df[col].fillna(0).values.astype('float32')
        uv = df[f'uaff_{col}'].fillna(0).values.astype('float32')
        uwv = df[f'uwaff_{col}'].fillna(0).values.astype('float32')
        cat_dot += uv * iv; cat_wdot += uwv * iv
        u_cn += uv**2; i_cn += iv**2
    
    dg = np.sqrt(u_gn * i_gn).clip(1e-8)
    dgw = np.sqrt(uw_gn * i_gn).clip(1e-8)
    dc = np.sqrt(u_cn * i_cn).clip(1e-8)
    
    df['genre_match_cos'] = genre_dot / dg
    df['genre_wmatch_cos'] = genre_wdot / dgw
    df['cat_match_cos'] = cat_dot / dc
    df['genre_match_dot'] = genre_dot
    df['cat_match_dot'] = cat_dot
    df['total_match_cos'] = (df['genre_match_cos'] + df['cat_match_cos']) / 2
    
    # Drop raw affinity & multi-hot
    drop_cols = [c for c in df.columns if c.startswith('uaff_') or c.startswith('uwaff_')]
    drop_cols += mh_cols
    df.drop(columns=drop_cols, inplace=True, errors='ignore')
    
    # ---- CONDITIONAL CROSS FEATURES ----
    print("    [7] Conditional cross features (user×genre target)...")
    
    # For each candidate, lookup user's avg target for each genre the candidate has
    cand_genres = df[['steamid', 'appid']].merge(
        item_feat_mh[['appid'] + mh_genre_cols], on='appid', how='left'
    )
    
    # For each row, find max/avg conditional target across candidate's genres
    cond_target_max = np.zeros(N, dtype=np.float32)
    cond_target_avg = np.zeros(N, dtype=np.float32)
    cond_pt_max = np.zeros(N, dtype=np.float32)
    cond_n_max = np.zeros(N, dtype=np.float32)
    
    # Build lookup dict for speed
    user_genre_dict = {}
    for _, row in user_genre_agg.iterrows():
        key = (row['steamid'], row['genre'])
        user_genre_dict[key] = (row['cond_genre_avg_target'], row['cond_genre_avg_pt'], row['cond_genre_n_games'])
    
    steamids_arr = df['steamid'].values
    for col in mh_genre_cols:
        genre_vals = cand_genres[col].fillna(0).values
        for i in range(N):
            if genre_vals[i] == 1:
                key = (steamids_arr[i], col)
                if key in user_genre_dict:
                    t, p, n = user_genre_dict[key]
                    if t > cond_target_max[i]:
                        cond_target_max[i] = t
                    cond_target_avg[i] += t
                    if p > cond_pt_max[i]:
                        cond_pt_max[i] = p
                    if n > cond_n_max[i]:
                        cond_n_max[i] = n
    
    # Normalize avg
    n_genres_per_item = cand_genres[mh_genre_cols].fillna(0).sum(axis=1).values.clip(1)
    cond_target_avg = cond_target_avg / n_genres_per_item
    
    df['cond_genre_target_max'] = cond_target_max
    df['cond_genre_target_avg'] = cond_target_avg
    df['cond_genre_pt_max'] = cond_pt_max
    df['cond_genre_n_max'] = cond_n_max
    df['cond_genre_pt_max_log'] = np.log1p(cond_pt_max)
    
    del cand_genres
    
    # ---- DEV/PUB CROSS ----
    print("    [8] Dev/Pub cross features...")
    df_d = df[['steamid', 'appid']].merge(dev_per_appid, on='appid', how='left')
    df_d['developer'] = df_d['developer'].apply(lambda x: x if isinstance(x, list) else [])
    df_d = df_d.explode('developer')
    df_d = df_d.merge(user_dev, on=['steamid', 'developer'], how='left')
    df_d['user_dev_n_games'] = df_d['user_dev_n_games'].fillna(0)
    df_d['user_dev_total_pt'] = df_d['user_dev_total_pt'].fillna(0)
    
    da = df_d.groupby(['steamid', 'appid']).agg(
        user_dev_max_games=('user_dev_n_games', 'max'),
        user_dev_sum_games=('user_dev_n_games', 'sum'),
        user_dev_max_pt=('user_dev_total_pt', 'max'),
        user_dev_has_history=('user_dev_n_games', lambda x: int((x > 0).any())),
    ).reset_index()
    df = df.merge(da, on=['steamid', 'appid'], how='left')
    df['user_dev_max_pt_log'] = np.log1p(df['user_dev_max_pt'].fillna(0))
    
    df_p = df[['steamid', 'appid']].merge(pub_per_appid, on='appid', how='left')
    df_p['publisher'] = df_p['publisher'].apply(lambda x: x if isinstance(x, list) else [])
    df_p = df_p.explode('publisher')
    df_p = df_p.merge(user_pub, on=['steamid', 'publisher'], how='left')
    df_p['user_pub_n_games'] = df_p['user_pub_n_games'].fillna(0)
    pa = df_p.groupby(['steamid', 'appid']).agg(
        user_pub_max_games=('user_pub_n_games', 'max'),
        user_pub_has_history=('user_pub_n_games', lambda x: int((x > 0).any())),
    ).reset_index()
    df = df.merge(pa, on=['steamid', 'appid'], how='left')
    
    # User-Dev TE
    df_dt = df[['steamid', 'appid']].merge(dev_exploded, on='appid', how='left')
    df_dt = df_dt.merge(user_dev_te[['steamid', 'developer', 'user_dev_te_smooth']], 
                         on=['steamid', 'developer'], how='left')
    dt_agg = df_dt.groupby(['steamid', 'appid']).agg(
        user_dev_te_max=('user_dev_te_smooth', 'max'),
        user_dev_te_mean=('user_dev_te_smooth', 'mean'),
    ).reset_index()
    df = df.merge(dt_agg, on=['steamid', 'appid'], how='left')
    
    # ---- INTERACTIONS ----
    print("    [9] Interaction features...")
    df['free_match'] = df['user_free_ratio'].fillna(0.5) * df['is_free'].fillna(0)
    df['als_x_centroid'] = df['als_score'] * df['centroid_sim']
    df['als_x_max_sim'] = df['als_score'] * df['max_sim_top10']
    df['als_x_dev_hist'] = df['als_score'] * df['user_dev_has_history'].fillna(0)
    df['als_dot_x_centroid'] = df['als_dot'] * df['centroid_sim']
    df['als_dot_x_max_sim'] = df['als_dot'] * df['max_sim_top10']
    df['als_x_genre_cos'] = df['als_score'] * df['genre_match_cos']
    df['als_x_total_match'] = df['als_score'] * df['total_match_cos']
    df['dev_te_x_als'] = df['item_dev_te_max'].fillna(0) * df['als_score']
    df['user_eng_x_item'] = df['user_avg_target_hist'].fillna(3) * df['item_avg_target_hist'].fillna(3)
    df['pop_vs_lib'] = df['item_n_players'].fillna(0) / df['user_total_games'].clip(lower=1)
    df['centroid_x_genre'] = df['centroid_sim'] * df['genre_match_cos']
    df['user_high_x_item_high'] = df['user_high_target_ratio'].fillna(0) * df['item_pct_high_target'].fillna(0)
    
    # Conditional × ALS
    df['als_x_cond_target'] = df['als_score'] * df['cond_genre_target_max']
    df['centroid_x_cond_target'] = df['centroid_sim'] * df['cond_genre_target_max']
    df['cond_target_x_item_target'] = df['cond_genre_target_max'] * df['item_avg_target_hist'].fillna(0)
    
    # ---- CLEANUP ----
    print("    [10] Cleanup...")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].fillna(0)
    for c in ['loccountrycode', 'type']:
        if c in df.columns:
            df[c] = df[c].fillna('unknown')
    
    print(f"    Final: {df.shape}")
    return df


# ============================================================
# BUILD v5
# ============================================================
print("\n" + "="*60)
print("Building v5 datasets")
print("="*60)

train_v5 = assemble_v5(train_sub.copy())
test_v5 = assemble_v5(test_sub.copy())

train_v5.sort_values('steamid', inplace=True)
test_v5.sort_values('steamid', inplace=True)

train_v5.to_parquet('artifacts/train_v5.parquet', index=False)
test_v5.to_parquet('artifacts/test_v5.parquet', index=False)

print(f"\nTrain v5: {train_v5.shape}")
print(f"Test v5:  {test_v5.shape}")
print(f"Train positive rate: {(train_v5['target']>0).mean():.4f}")


All loaded!

[1] Negative subsampling: top-100 by ALS rank + all positives
Train: 1064400 → 358629 (33.7%)
Test:  266100 → 89741 (100.0%)
Train positive rate: 0.0241 (was 0.0081)
Test positive rate:  0.0248

[2] PCA on ALS factor products
PCA explained variance: 0.2997 (24 components)
Per component: [0.0233 0.021  0.0168 0.0156 0.0149]

[3] Conditional user features
User-Genre conditional stats: 56447 pairs
User-Category conditional stats: 22599 pairs

[4] Playtime-weighted affinity...
All precomputed!

Building v5 datasets

  Assembling v5, rows=358629
    [1] ALS score features...
    [2] ALS PCA factor products...
    [3] Item-item similarity...
      Top-K similarity...
    [4] User features...
    [5] Item features...
    [6] Genre/Category match...
    [7] Conditional cross features (user×genre target)...
    [8] Dev/Pub cross features...
    [9] Interaction features...
    [10] Cleanup...
    Final: (358629, 138)

  Assembling v5, rows=89741
    [1] ALS score features...
    [2]

In [3]:
import pandas as pd
import numpy as np
from catboost import CatBoostRanker, Pool
from catboost.utils import get_gpu_device_count
import mlflow

print(f"GPU: {get_gpu_device_count()}")

train = pd.read_parquet("artifacts/train_v5.parquet").sort_values("steamid")
test = pd.read_parquet("artifacts/test_v5.parquet").sort_values("steamid")

drop_cols = ["steamid", "appid", "target"]
cat_features = ["loccountrycode", "type"]

for cf in cat_features:
    train[cf] = train[cf].fillna("unknown").astype(str)
    test[cf] = test[cf].fillna("unknown").astype(str)

feature_cols = [c for c in train.columns if c not in drop_cols]
print(f"Features: {len(feature_cols)}")

X_train, y_train, q_train = train[feature_cols], train["target"], train["steamid"]
X_test, y_test, q_test = test[feature_cols], test["target"], test["steamid"]

train_pool = Pool(data=X_train, label=y_train, group_id=q_train, cat_features=cat_features)
test_pool = Pool(data=X_test, label=y_test, group_id=q_test, cat_features=cat_features)

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Steam_RecSys_v5")

results = {}

# ============================================================
# Run 1: YetiRankPairwise (наш лучший loss)
# ============================================================
for loss_name, loss_params in [
    ("YetiRankPairwise", {"loss_function": "YetiRankPairwise", "depth": 7}),
    ("PairLogitPairwise", {"loss_function": "PairLogitPairwise", "depth": 6, "border_count": 128, "bootstrap_type": "Bayesian"}),
    ("YetiRank", {"loss_function": "YetiRank", "depth": 7}),
]:
    print(f"\n{'='*60}")
    print(f"Training: {loss_name}")
    print('='*60)
    
    with mlflow.start_run(run_name=f"v5_{loss_name}"):
        params = {
            "iterations": 5000,
            "learning_rate": 0.03,
            "l2_leaf_reg": 3.0,
            "custom_metric": ["NDCG:top=10", "MAP:top=10"],
            "eval_metric": "NDCG:top=10",
            "early_stopping_rounds": 200,
            "random_seed": 42,
            "task_type": "GPU",
            "devices": "0",
            "verbose": 200,
        }
        params.update(loss_params)
        
        mlflow.log_params(params)
        model = CatBoostRanker(**params)
        model.fit(train_pool, eval_set=test_pool)
        
        score = model.get_best_score()["validation"]["NDCG:top=10;type=Base"]
        best_iter = model.get_best_iteration()
        print(f"{loss_name}: NDCG@10 = {score:.4f} (iter {best_iter})")
        
        mlflow.log_metric("best_ndcg_10", score)
        model.save_model(f"artifacts/catboost_v5_{loss_name.lower()}.cbm")
        results[loss_name] = (score, model, best_iter)

# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*60)
print("RESULTS SUMMARY")
print("="*60)
print(f"v2 baseline:       NDCG@10 = 0.3804")
print(f"v3 PairLogitPW:    NDCG@10 = 0.4121")
print(f"v4 YetiRankPW:     NDCG@10 = 0.4230")
for name, (score, _, iter_) in results.items():
    print(f"v5 {name:20s}: NDCG@10 = {score:.4f} (iter {iter_})")

# Feature importance
best_name = max(results, key=lambda x: results[x][0])
best_model = results[best_name][1]

fi = best_model.get_feature_importance(train_pool)
fi_df = pd.DataFrame({'feature': feature_cols, 'importance': fi}).sort_values('importance', ascending=False)

print(f"\nTop-50 features ({best_name}):")
print(fi_df.head(50).to_string(index=False))
fi_df.to_csv('artifacts/feature_importance_v5.csv', index=False)


GPU: 1
Features: 135


2026/05/15 22:47:30 INFO mlflow.tracking.fluent: Experiment with name 'Steam_RecSys_v5' does not exist. Creating a new experiment.



Training: YetiRankPairwise
Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because MAP, NDCG is/are not implemented for GPU
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric MAP:top=10 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.5235984	best: 0.5235984 (0)	total: 129ms	remaining: 10m 42s
200:	test: 0.6715926	best: 0.6716267 (199)	total: 21.7s	remaining: 8m 39s
400:	test: 0.6905601	best: 0.6905601 (400)	total: 43.4s	remaining: 8m 17s
600:	test: 0.6999912	best: 0.7005210 (599)	total: 1m 5s	remaining: 7m 56s
800:	test: 0.7029159	best: 0.7031910 (788)	total: 1m 26s	remaining: 7m 35s
1000:	test: 0.7053437	best: 0.7059934 (993)	total: 1m 48s	remaining: 7m 14s
bestTest = 0.7059933645
bestIteration = 993
Shrink model to first 994 iterations.
YetiRankPairwise: NDCG@10 = 0.7060 (iter 993)

Training: PairLogitPairwise
Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.5989740	best: 0.5989740 (0)	total: 52.8ms	remaining: 4m 24s


Default metric period is 5 because MAP, NDCG is/are not implemented for GPU
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric MAP:top=10 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


200:	test: 0.7105064	best: 0.7105064 (200)	total: 7.21s	remaining: 2m 52s
400:	test: 0.7160480	best: 0.7162855 (399)	total: 14.3s	remaining: 2m 44s
600:	test: 0.7201437	best: 0.7211182 (590)	total: 21.5s	remaining: 2m 37s
800:	test: 0.7221675	best: 0.7224406 (752)	total: 28.6s	remaining: 2m 29s
1000:	test: 0.7247507	best: 0.7251870 (996)	total: 35.7s	remaining: 2m 22s
1200:	test: 0.7274945	best: 0.7293244 (1142)	total: 42.9s	remaining: 2m 15s
bestTest = 0.7293243603
bestIteration = 1142
Shrink model to first 1143 iterations.
PairLogitPairwise: NDCG@10 = 0.7293 (iter 1142)

Training: YetiRank
Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.6167324	best: 0.6167324 (0)	total: 29.7ms	remaining: 2m 28s


Default metric period is 5 because MAP, NDCG is/are not implemented for GPU
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric MAP:top=10 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


200:	test: 0.6926882	best: 0.6926882 (200)	total: 3.13s	remaining: 1m 14s
400:	test: 0.7032407	best: 0.7038035 (398)	total: 6.22s	remaining: 1m 11s
600:	test: 0.7079683	best: 0.7079683 (600)	total: 9.3s	remaining: 1m 8s
800:	test: 0.7107124	best: 0.7110289 (783)	total: 12.4s	remaining: 1m 4s
1000:	test: 0.7143028	best: 0.7143828 (980)	total: 15.5s	remaining: 1m 1s
1200:	test: 0.7142630	best: 0.7149751 (1169)	total: 18.5s	remaining: 58.6s
1400:	test: 0.7165984	best: 0.7165984 (1400)	total: 21.6s	remaining: 55.5s
1600:	test: 0.7169901	best: 0.7175549 (1563)	total: 24.7s	remaining: 52.4s
bestTest = 0.7175548685
bestIteration = 1563
Shrink model to first 1564 iterations.
YetiRank: NDCG@10 = 0.7176 (iter 1563)

RESULTS SUMMARY
v2 baseline:       NDCG@10 = 0.3804
v3 PairLogitPW:    NDCG@10 = 0.4121
v4 YetiRankPW:     NDCG@10 = 0.4230
v5 YetiRankPairwise    : NDCG@10 = 0.7060 (iter 993)
v5 PairLogitPairwise   : NDCG@10 = 0.7293 (iter 1142)
v5 YetiRank            : NDCG@10 = 0.7176 (iter 1563)

In [5]:
import pandas as pd
import numpy as np
from catboost import CatBoostRanker, Pool

# ============================================================
# 1. ПРОВЕРКА: размеры групп в тесте
# ============================================================
test_v4 = pd.read_parquet("artifacts/test_v3.parquet")
test_v5 = pd.read_parquet("artifacts/test_v5.parquet")

print("="*60)
print("ПРОВЕРКА РАЗМЕРОВ ТЕСТОВЫХ ГРУПП")
print("="*60)

v4_groups = test_v4.groupby('steamid').size()
v5_groups = test_v5.groupby('steamid').size()

print(f"\nv4 test: {len(test_v4)} rows, {test_v4['steamid'].nunique()} users")
print(f"  Candidates per user: mean={v4_groups.mean():.0f}, min={v4_groups.min()}, max={v4_groups.max()}")
print(f"  Positive rate: {(test_v4['target']>0).mean():.4f}")

print(f"\nv5 test: {len(test_v5)} rows, {test_v5['steamid'].nunique()} users")
print(f"  Candidates per user: mean={v5_groups.mean():.0f}, min={v5_groups.min()}, max={v5_groups.max()}")
print(f"  Positive rate: {(test_v5['target']>0).mean():.4f}")

# ============================================================
# 2. СОБИРАЕМ v5 ФИЧИ ДЛЯ ПОЛНОГО ТЕСТА (300 кандидатов)
# ============================================================
print("\n" + "="*60)
print("Собираем v5 features для полного теста (300 кандидатов)")
print("="*60)

test_full_base = pd.read_parquet('artifacts/test_reranker_base.parquet')
test_full_base = test_full_base.rename(columns={'score': 'als_score', 'rank': 'als_rank'})
print(f"Full test: {len(test_full_base)} rows, {test_full_base['steamid'].nunique()} users")

test_v5_full = assemble_v5(test_full_base.copy())
test_v5_full.sort_values('steamid', inplace=True)
print(f"Test v5 full: {test_v5_full.shape}")

# ============================================================
# 3. EVAL НА ПОЛНОМ ТЕСТЕ
# ============================================================
print("\n" + "="*60)
print("EVALUATION НА ПОЛНОМ ТЕСТЕ (300 кандидатов)")
print("="*60)

drop_cols = ["steamid", "appid", "target"]
cat_features = ["loccountrycode", "type"]

for cf in cat_features:
    test_v5_full[cf] = test_v5_full[cf].fillna("unknown").astype(str)

feature_cols = [c for c in test_v5_full.columns if c not in drop_cols]

test_pool_full = Pool(
    data=test_v5_full[feature_cols],
    label=test_v5_full["target"],
    group_id=test_v5_full["steamid"],
    cat_features=cat_features
)

for model_name, model_path in [
    ("PairLogitPairwise", "artifacts/catboost_v5_pairlogitpairwise.cbm"),
    ("YetiRankPairwise", "artifacts/catboost_v5_yetirankpairwise.cbm"),
    ("YetiRank", "artifacts/catboost_v5_yetirank.cbm"),
]:
    model = CatBoostRanker()
    model.load_model(model_path)
    
    metrics = model.eval_metrics(
        test_pool_full,
        metrics=["NDCG:top=10"],
    )
    
    # Посмотрим какие ключи доступны
    print(f"\n{model_name}:")
    print(f"  Available metric keys: {list(metrics.keys())}")
    
    # Берём первый подходящий ключ
    ndcg_key = [k for k in metrics.keys() if 'NDCG' in k][0]
    ndcg_values = metrics[ndcg_key]
    
    best_iter = model.get_best_iteration()
    if best_iter is not None and best_iter < len(ndcg_values):
        ndcg = ndcg_values[best_iter]
    else:
        ndcg = max(ndcg_values)
    
    print(f"  Subsampled test (top-100):  из обучения")
    print(f"  ПОЛНЫЙ test (300 канд):     NDCG@10 = {ndcg:.4f}")

print("\n" + "="*60)
print("ЧЕСТНОЕ СРАВНЕНИЕ (все на 300 кандидатах)")
print("="*60)
print(f"v2 baseline:        NDCG@10 = 0.3804")
print(f"v3 PairLogitPW:     NDCG@10 = 0.4121")
print(f"v4 YetiRankPW:      NDCG@10 = 0.4230")
print(f"v5 модели:          см. выше ↑")


ПРОВЕРКА РАЗМЕРОВ ТЕСТОВЫХ ГРУПП

v4 test: 266100 rows, 887 users
  Candidates per user: mean=300, min=300, max=300
  Positive rate: 0.0084

v5 test: 89741 rows, 887 users
  Candidates per user: mean=101, min=100, max=117
  Positive rate: 0.0248

Собираем v5 features для полного теста (300 кандидатов)
Full test: 266100 rows, 887 users

  Assembling v5, rows=266100
    [1] ALS score features...
    [2] ALS PCA factor products...
    [3] Item-item similarity...
      Top-K similarity...
    [4] User features...
    [5] Item features...
    [6] Genre/Category match...
    [7] Conditional cross features (user×genre target)...
    [8] Dev/Pub cross features...
    [9] Interaction features...
    [10] Cleanup...
    Final: (266100, 138)
Test v5 full: (266100, 138)

EVALUATION НА ПОЛНОМ ТЕСТЕ (300 кандидатов)

PairLogitPairwise:
  Available metric keys: ['NDCG:top=10;type=Base']
  Subsampled test (top-100):  из обучения
  ПОЛНЫЙ test (300 канд):     NDCG@10 = 0.2204

YetiRankPairwise:
  Avail